# Snorkel Finance: Aligning Trace Splits with Online Env Splits

**Goal**: Create `act_prm` and `act_lm` configs that reference the reasoning traces
at `mzio/aprm-snorkelai_agent_finance_reasoning`, such that train/eval/test splits
align with the online `snorkel_finance` environment's splits.

**Problem**: The online env uses HuggingFace `train_test_split()` on the original
`snorkelai/agent-finance-reasoning` dataset (357 rows), while `ActPrmEnv` and `ActionLmEnv`
use numpy-based shuffling on trajectory indices. These produce different splits.

**Key insight**: `unique_data_sample_id` in the traces == row index in the original dataset.
Verified: `original_ds[uid]['answer'] == trace_row['answer']` for all 357 UIDs.

**Important**: The traces dataset has 3 HF splits:
- `train`: 4610 rows, 357 tasks, all `generation_id=0` (base traces from 5 models)
- `aprm_sft_genthinkact_*_b019`: 2825 rows, 51 tasks, 234 unique generation_ids (Act-PRM rollouts)
- `*_b019_gen0`: 688 rows, 51 tasks, `generation_id=0` only (filtered subset of above)

We only use the `train` split to avoid duplicates and non-zero generation_ids.

**Approach**:
1. Add an index column to the original dataset, reproduce `_get_splits()` → get index→split mapping
2. Use that mapping to assign each trace row (from `train` split only) to the correct split
3. Push a new HF dataset (`-aligned`) with proper train/eval/test HF splits
4. Create configs that load the properly-split dataset

**Result**: Pushed `mzio/aprm-snorkelai_agent_finance_reasoning-aligned` with:
- train: 3768 rows (290 tasks)
- eval: 383 rows (30 tasks)
- test: 459 rows (37 tasks)

All rows have `generation_id=0`. Split UIDs match `configs/environments/snorkel_finance/default.yaml` (seed=0) exactly.

In [ ]:
from datasets import load_dataset, DatasetDict, Dataset

## 1. Build index→split mapping from the online SnorkelFinanceEnv

The online env (`configs/environments/snorkel_finance/default.yaml`) loads
`snorkelai/agent-finance-reasoning` (357 rows) and splits via `_get_splits()`:
1. `train_test_split(test_size=37, seed=0)` → test (37) + remainder (320)
2. `train_test_split(test_size=30, seed=0)` on remainder → eval (30) + train (290)

In [ ]:
# Load original dataset, add index column for tracking through splits
original_ds = load_dataset("snorkelai/agent-finance-reasoning", split="train")
print(f"Original dataset: {len(original_ds)} rows, {len(set(original_ds['id']))} unique query_ids")

indexed_ds = original_ds.map(
    lambda _, idx: {"_orig_idx": idx},
    with_indices=True,
    remove_columns=original_ds.column_names,
    load_from_cache_file=False,
)

# Reproduce _get_splits() from SnorkelFinanceEnv (default.yaml: seed=0, test=37, val=30)
SEED = 0
trainval_test = indexed_ds.train_test_split(test_size=37, shuffle=True, seed=SEED)
train_val = trainval_test["train"].train_test_split(test_size=30, shuffle=True, seed=SEED)

# Build index → split mapping
index_to_split = {}
for split_name, split_ds in [("train", train_val["train"]), ("eval", train_val["test"]), ("test", trainval_test["test"])]:
    for row in split_ds:
        index_to_split[row["_orig_idx"]] = split_name

from collections import Counter
print(f"Index→split mapping: {dict(Counter(index_to_split.values()))}")

## 2. Verify: `unique_data_sample_id` == row index in original dataset

In [ ]:
# Load traces and verify uid == index by answer matching
traces_ds = load_dataset("mzio/aprm-snorkelai_agent_finance_reasoning")
print("Traces splits:")
for name, ds in traces_ds.items():
    uids = sorted(set(ds["unique_data_sample_id"]))
    print(f"  {name}: {len(ds)} rows, {len(uids)} tasks, uid range [{min(uids)}, {max(uids)}]")

# Spot-check: does original_ds[uid].answer == trace answer?
for uid in [0, 1, 50, 100, 200, 356]:
    trace_rows = [r for r in traces_ds["train"] if r["unique_data_sample_id"] == uid]
    if trace_rows:
        match = trace_rows[0]["answer"].strip() == original_ds[uid]["answer"].strip()
        print(f"uid={uid}: answer_match={match}, company={original_ds[uid]['company']}")

## 3. Create aligned dataset with proper HF splits

**Only use the `train` HF split** from the traces dataset. The other splits
(`aprm_sft_genthinkact_*`, `*_gen0`) contain Act-PRM rollouts with non-zero
`generation_id`s and duplicate rows for 51 tasks — including them would cause
duplicates and break the act_lm `_maybe_build_full_states()` logic.

In [ ]:
# Only use the 'train' HF split (357 tasks, all generation_id=0)
# The other splits contain Act-PRM rollouts with non-zero generation_ids and duplicates
train_split = traces_ds["train"]
print(f"Using 'train' split only: {len(train_split)} rows, "
      f"{len(set(train_split['unique_data_sample_id']))} tasks, "
      f"generation_ids: {sorted(set(train_split['generation_id']))}")

# Assign each trace row to the online env's split via unique_data_sample_id → index_to_split
split_buckets = {"train": [], "eval": [], "test": []}
for i in range(len(train_split)):
    row = dict(train_split[i])
    uid = row["unique_data_sample_id"]
    online_split = index_to_split.get(uid)
    if online_split:
        split_buckets[online_split].append(row)

aligned_ds = DatasetDict({
    name: Dataset.from_list(rows) for name, rows in split_buckets.items() if rows
})
print(aligned_ds)
for name, ds in aligned_ds.items():
    uids = set(ds["unique_data_sample_id"])
    print(f"  {name}: {len(ds)} rows, {len(uids)} tasks")

In [ ]:
# Push to HuggingFace Hub
DS_NAME = "mzio/aprm-snorkelai_agent_finance_reasoning-aligned"
aligned_ds.push_to_hub(DS_NAME)
print(f"Pushed to: https://huggingface.co/datasets/{DS_NAME}")

## 4. Verify split alignment

Confirm the aligned dataset's UIDs exactly match the online env's splits.

In [ ]:
# Reload aligned dataset from Hub and verify against online env
aligned_ds = load_dataset("mzio/aprm-snorkelai_agent_finance_reasoning-aligned")
aligned_train_uids = set(aligned_ds["train"]["unique_data_sample_id"])
aligned_eval_uids = set(aligned_ds["eval"]["unique_data_sample_id"])
aligned_test_uids = set(aligned_ds["test"]["unique_data_sample_id"])

# Reproduce online env splits
online_train_uids = set(r["_orig_idx"] for r in train_val["train"])
online_eval_uids = set(r["_orig_idx"] for r in train_val["test"])
online_test_uids = set(r["_orig_idx"] for r in trainval_test["test"])

print("UID sets match online env splits:")
print(f"  train: {aligned_train_uids == online_train_uids} ({len(aligned_train_uids)} tasks)")
print(f"  eval:  {aligned_eval_uids == online_eval_uids} ({len(aligned_eval_uids)} tasks)")
print(f"  test:  {aligned_test_uids == online_test_uids} ({len(aligned_test_uids)} tasks)")

# Verify disjointness
print(f"\nSplits disjoint:")
print(f"  train ∩ eval: {len(aligned_train_uids & aligned_eval_uids)} (should be 0)")
print(f"  train ∩ test: {len(aligned_train_uids & aligned_test_uids)} (should be 0)")
print(f"  eval ∩ test:  {len(aligned_eval_uids & aligned_test_uids)} (should be 0)")
print(f"  total tasks:  {len(aligned_train_uids | aligned_eval_uids | aligned_test_uids)} (should be 357)")

# Verify all answers match
all_ok = True
for split_name, split_ds in aligned_ds.items():
    for uid in set(split_ds["unique_data_sample_id"]):
        trace_row = next(r for r in split_ds if r["unique_data_sample_id"] == uid)
        if trace_row["answer"].strip() != original_ds[uid]["answer"].strip():
            print(f"MISMATCH: {split_name}, uid={uid}")
            all_ok = False
print(f"\nAll answers match original dataset: {all_ok}")

## 5. Config Notes

**Aligned dataset**: `mzio/aprm-snorkelai_agent_finance_reasoning-aligned`

Only uses the `train` HF split from the source traces (all `generation_id=0`).
The other splits (`aprm_sft_genthinkact_*`, `*_gen0`) contain Act-PRM rollouts
with 234 unique non-zero generation_ids for 51 tasks — excluded to avoid duplicates.

- `train` HF split: 3768 rows, 290 tasks (matches online env train)
- `eval` HF split: 383 rows, 30 tasks (matches online env eval)
- `test` HF split: 459 rows, 37 tasks (matches online env test)

**Configs created**:
- `configs/environments/act_prm/snorkel_finance_aligned.yaml` — loads `train` split, sub-splits into 250 train + 40 val via numpy shuffle
- `configs/environments/act_lm/snorkel_finance_aligned.yaml` — loads `train` split, sub-splits 85/15 via `frac_train_tasks`

**Eval with online env**: Use `configs/environments/snorkel_finance/default.yaml` with `split: "eval"` or `split: "test"` — the task UIDs will match exactly since both use `seed=0` on the same original dataset.

**Note on question-level overlap**: The original dataset has 79 unique questions, each with ~4-5 model runs (o3, gemini-2.5-pro, claude-opus-4, claude-sonnet-4, o4-mini). The online env splits at the *row* level (not question level), so the same question can appear in multiple splits via different model runs. This is inherent to the online env's design.